# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access the metadata as an object and print core information
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Let's print all record sets with their '@id', 'name', and all fields
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: @id={rs.id}, name={rs.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id={field.id}, name={field.name}, type={field.data_type}")
    print()
# For further processing, we'll use @ids to reference sets and fields.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
# (Fill in with available record set @ids and field @ids from previous cell output)

record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
# Example: Pick the first record set for further exploration
if len(record_sets_ids) > 0:
    main_record_set_id = record_sets_ids[0]
    print(f"Preview of record set '@id': {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a numeric field for analysis
# We'll choose the first numeric field found in the first record set
record_set = dataset.metadata.record_sets[0]
numeric_field_id = None
group_field_id = None
for f in record_set.fields:
    if f.data_type in ['schema:Float', 'schema:Number', 'schema:Integer'] and numeric_field_id is None:
        numeric_field_id = f.id
    # Select first non-numeric field for grouping if suitable
    if f.data_type in ['schema:Text', 'schema:String'] and group_field_id is None:
        group_field_id = f.id

df = dataframes[record_set.id]

if numeric_field_id and numeric_field_id in df.columns:
    # Convert numeric field to float (in case)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean): {filtered_df.shape[0]} rows.")
    display(filtered_df.head())

    # Normalize the numeric field for filtered records
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optionally group by another text field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field if available
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped data is available, show a barplot
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded and explored the FAIR² dataset:
  - Reviewed its record sets and fields using Croissant `@id` conventions.
  - Loaded tabular data for each record set and inspected its schema.
  - Performed basic exploratory data analysis, including filtering and normalization of numeric fields, and grouped summary statistics.
  - Visualized data distributions using `seaborn` and `matplotlib`.
  
This workflow can be used to explore, analyze, and validate any dataset that provides a [Croissant schema](https://mlcommons.org/croissant/).